In [ ]:
import tensorflow as tf
import os
import glob
import numpy as np
from model import build_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, tf_load_sample_from_files
cnn_model=build_cnn_model(INPUT_SHAPE,OUTPUT_DIM_NOTES,False)
cnn_model.summary()
tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch02_valAcc0.5887.weights.h5')#('guitarmidi.weights.h5')

input_data_dir = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices/training_subset'
input_filepaths = sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', '*.npy'), recursive=True))
train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
train_dataset=train_dataset.take(100)
train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
  """A generator function that yields input data for quantization."""
  for input_value in train_dataset.batch(1).prefetch(tf.data.AUTOTUNE):
    # TFLite converter expects a list of input tensors
    yield [input_value[0]]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)

I0000 00:00:1762976789.500094   65868 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1762976789.525891   65868 cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1762976790.028680   65868 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1762976790.618079   65868 gpu_device.cc:2456] TensorFlow was not built with CUDA kernel bi

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 312, 256, 32)   │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ (None, 312, 256, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 312, 256, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 312, 256, 64)   │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ (None, 312, 256, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 312, 256, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 312, 256, 128)  │       204,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_2 (Activation)       │ (None, 312, 256, 128)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 312, 256, 128)  │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 39, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lambda (Lambda)                 │ (None, 768)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 89)             │        68,441 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 326,361 (1.24 MB)

 Trainable params: 325,913 (1.24 MB)

 Non-trainable params: 448 (1.75 KB)

INFO:tensorflow:Assets written to: /tmp/tmpicfw_vnc/assets


INFO:tensorflow:Assets written to: /tmp/tmpicfw_vnc/assets


Saved artifact at '/tmp/tmpicfw_vnc'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 312, 256, 1), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 89), dtype=tf.float32, name=None)
Captures:
  139588903479696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139588903480848: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139588903481424: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139588903481040: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139588903480080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139588903481232: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139588903481808: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139588903482768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139588903483536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139588903483728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1395889034823

/home/gerald/anaconda3/envs/gpu/lib/python3.13/site-packages/tensorflow/lite/python/convert.py:863: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1762976792.272973   65868 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1762976792.272989   65868 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1762976792.273180   65868 reader.cc:83] Reading SavedModel from: /tmp/tmpicfw_vnc
I0000 00:00:1762976792.273646   65868 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1762976792.273653   65868 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpicfw_vnc
I0000 00:00:1762976792.277569   65868 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1762976792.278462   65868 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1762976792.305112   65868 loader.cc:220] Running initialization op on S